# Notebook 03 — Modelos Predictivos de Demanda

**Objetivo:** Desarrollar y comparar modelos supervisados (XGBoost, LSTM) y el baseline no supervisado (K-Means clustering) para estimar la demanda diaria por máquina expendedora.  
**Datos:** Snapshots de QuickBase — 271 789 transacciones, 405 máquinas, El Paso TX.

### Métricas de evaluación

| Métrica | Fórmula | Qué mide |
|---|---|---|
| **MAE** | mean\|y − ŷ\| | Error absoluto promedio en dólares |
| **RMSE** | √mean(y − ŷ)² | Igual que MAE pero penaliza errores grandes |
| **MdAE** | median\|y − ŷ\| | Mediana del error — robusta a máquinas outlier |
| **R²** | 1 − SS_res/SS_tot | Fracción de varianza explicada (1 = perfecto) |
| **SMAPE** | mean(\|y−ŷ\|/((\|y\|+\|ŷ\|)/2))·100 | Error porcentual simétrico; tolera días sin venta |
| **Bias** | mean(ŷ − y) | Error sistemático: positivo = sobreestima |

---

## Estructura
1. Setup y carga de datos
2. Ingeniería de características
3. División cronológica train / test
4. Modelo XGBoost
5. Modelo LSTM
6. Baseline K-Means (perfiles de clúster)
7. Comparación y conclusiones
8. Persistencia de artefactos

## 0 · Setup

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.patches import FancyArrowPatch

from src.features.feature_engineering import (
    build_feature_matrix, time_split,
    FEATURE_COLS, LSTM_COLS, TARGET_COL,
    load_clusters,
)
import src.models.xgboost_demand      as xgb_model
import src.models.lstm_demand          as lstm_model
import src.models.clustering_profiles  as cluster_model

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

SNAPSHOTS_DIR  = ROOT / 'data' / 'snapshots'
CLUSTERS_PATH  = ROOT / 'data' / 'vm_clusters.csv'
FIGURES_DIR    = ROOT / 'reports' / 'figures'
METRICS_DIR    = ROOT / 'reports' / 'metrics'
EXPORTS_DIR    = ROOT / 'reports' / 'exports'
ARTIFACTS_DIR  = ROOT / 'models_artifacts'

for d in [FIGURES_DIR, METRICS_DIR, EXPORTS_DIR,
          ARTIFACTS_DIR/'xgboost', ARTIFACTS_DIR/'lstm', ARTIFACTS_DIR/'clustering']:
    d.mkdir(parents=True, exist_ok=True)

print('Setup completo ✓')

## 1 · Ingeniería de características

| Grupo | Características |
|---|---|
| Temporales | `dow`, `month`, `is_weekend`, `week_of_year` |
| Rezagos | `lag_1`, `lag_7`, `lag_14` |
| Estadísticas móviles | `roll_mean_7/14`, `roll_std_7/14` |
| Clúster | `cluster` — perfil K-Means (0–3) |

In [ ]:
daily = build_feature_matrix(
    snapshots_dir=str(SNAPSHOTS_DIR),
    clusters_path=str(CLUSTERS_PATH),
)
clusters_df = load_clusters(str(CLUSTERS_PATH))

print(f'Filas totales : {len(daily):,}')
print(f'Máquinas      : {daily["vm_control"].nunique():,}')
print(f'Rango fechas  : {daily["date"].min().date()} → {daily["date"].max().date()}')
print(f'Días totales  : {daily["date"].nunique()}')
daily.head(3)

## 2 · División cronológica train / test (80 / 20)

In [ ]:
train_df, test_df, cutoff = time_split(daily, test_frac=0.20)

print(f'Fecha de corte : {pd.Timestamp(cutoff).date()}')
print(f'Train          : {len(train_df):,} filas  |  {train_df["date"].nunique()} días únicos')
print(f'Test           : {len(test_df):,} filas  |  {test_df["date"].nunique()} días únicos')

fig, ax = plt.subplots(figsize=(13, 3))
daily_agg = daily.groupby('date')['total_sale'].sum().reset_index()
ax.fill_between(daily_agg['date'], daily_agg['total_sale'],
                where=(daily_agg['date'] < cutoff), alpha=0.6, label='Train', color='steelblue')
ax.fill_between(daily_agg['date'], daily_agg['total_sale'],
                where=(daily_agg['date'] >= cutoff), alpha=0.6, label='Test', color='tomato')
ax.axvline(cutoff, color='black', linestyle='--', lw=1.5, label=f'Corte: {pd.Timestamp(cutoff).date()}')
ax.set_title('Ventas diarias totales — división Train / Test')
ax.set_xlabel('Fecha'); ax.set_ylabel('Ventas ($)')
ax.legend(); plt.tight_layout()
plt.savefig(FIGURES_DIR / 'train_test_split.png', bbox_inches='tight')
plt.show()

## 3 · Modelo XGBoost

Gradient Boosted Trees sobre 12 características tabulares por fila-día.  
Early stopping activo (30 rondas sin mejora sobre validación interna).

In [ ]:
train_clean = train_df.dropna(subset=FEATURE_COLS + [TARGET_COL])
test_clean  = test_df.dropna(subset=FEATURE_COLS + [TARGET_COL])

val_dates  = sorted(train_clean['date'].unique())
val_cutoff = val_dates[int(len(val_dates) * 0.90)]
val_mask   = train_clean['date'] >= val_cutoff

X_tr = train_clean.loc[~val_mask, FEATURE_COLS]
y_tr = train_clean.loc[~val_mask, TARGET_COL]
X_va = train_clean.loc[ val_mask, FEATURE_COLS]
y_va = train_clean.loc[ val_mask, TARGET_COL]
X_te = test_clean[FEATURE_COLS]
y_te = test_clean[TARGET_COL]

print(f'Train efectivo : {len(X_tr):,}  |  Validación: {len(X_va):,}  |  Test: {len(X_te):,}')

xgb = xgb_model.train(X_tr, y_tr, X_va, y_va)

y_pred_xgb, metrics_xgb = xgb_model.evaluate(xgb, X_te, y_te)
metrics_xgb_by_cluster  = xgb_model.evaluate_by_cluster(
    xgb, X_te, y_te, test_clean['cluster'].reset_index(drop=True)
)

print('\nXGBoost — métricas globales:')
for k, v in metrics_xgb.items():
    print(f'  {k:12s}: {v}')
print('\nMétricas por clúster:')
print(metrics_xgb_by_cluster.to_string(index=False))

In [ ]:
fi = pd.Series(xgb.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fi.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Importancia de características — XGBoost')
axes[0].set_xlabel('Importancia (gain)')

sample_idx = np.random.choice(len(y_te), size=min(500, len(y_te)), replace=False)
axes[1].scatter(y_te.values[sample_idx], y_pred_xgb[sample_idx],
                alpha=0.4, s=20, color='steelblue')
lims = [0, max(y_te.max(), y_pred_xgb.max()) * 1.05]
axes[1].plot(lims, lims, 'r--', lw=1.5, label='Línea ideal')
axes[1].set_xlim(lims); axes[1].set_ylim(lims)
axes[1].set_title(f'Pred vs Real — XGBoost  (R²={metrics_xgb["R2"]:.3f})')
axes[1].set_xlabel('Ventas reales ($)'); axes[1].set_ylabel('Ventas predichas ($)')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'xgboost_evaluation.png', bbox_inches='tight')
plt.show()

In [ ]:
# Evolución temporal agregada
test_agg = test_clean[['date', TARGET_COL]].copy()
test_agg['pred_xgb'] = y_pred_xgb
daily_pred_xgb = test_agg.groupby('date').sum()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily_pred_xgb.index, daily_pred_xgb[TARGET_COL], label='Real', lw=2)
ax.plot(daily_pred_xgb.index, daily_pred_xgb['pred_xgb'], label='XGBoost', lw=1.8, linestyle='--')
ax.set_title('Ventas diarias totales — período de prueba (XGBoost)')
ax.set_xlabel('Fecha'); ax.set_ylabel('Ventas ($)')
ax.legend(); plt.tight_layout()
plt.savefig(FIGURES_DIR / 'xgboost_timeseries.png', bbox_inches='tight')
plt.show()

## 4 · Modelo LSTM

Red LSTM de dos capas entrenada globalmente con secuencias de 14 días.  
MinMaxScaler ajustado **solo** sobre entrenamiento para evitar fuga de datos.

```
Input (14, 4) → LSTM(64) → Dropout(0.2) → LSTM(32) → Dropout(0.2) → Dense(16) → Dense(1)
```

In [ ]:
SEQ_LEN    = lstm_model.SEQ_LEN
lstm_feats = [c for c in LSTM_COLS if c != TARGET_COL]

train_lstm = train_df[['vm_control', 'date', TARGET_COL] + lstm_feats].copy()
test_lstm  = test_df[ ['vm_control', 'date', TARGET_COL] + lstm_feats].copy()

X_train_seq, y_train_seq, scaler_lstm = lstm_model.create_sequences(
    train_lstm, feature_cols=lstm_feats, target_col=TARGET_COL,
    seq_len=SEQ_LEN, fit_scaler=True,
)

last_train_dates = sorted(train_df['date'].unique())[-SEQ_LEN:]
context = train_df[train_df['date'].isin(last_train_dates)][['vm_control', 'date', TARGET_COL] + lstm_feats]
test_with_ctx = pd.concat([context, test_lstm], ignore_index=True)

X_test_seq, y_test_seq, _ = lstm_model.create_sequences(
    test_with_ctx, feature_cols=lstm_feats, target_col=TARGET_COL,
    seq_len=SEQ_LEN, scaler=scaler_lstm, fit_scaler=False,
)

print(f'Train: {X_train_seq.shape}  |  Test: {X_test_seq.shape}')

In [ ]:
n_val      = int(len(X_train_seq) * 0.10)
X_val_seq  = X_train_seq[-n_val:]
y_val_seq  = y_train_seq[-n_val:]
X_trn_seq  = X_train_seq[:-n_val]
y_trn_seq  = y_train_seq[:-n_val]

n_lstm_cols = 1 + len(lstm_feats)
lstm = lstm_model.build_model(input_shape=(SEQ_LEN, n_lstm_cols))
lstm.summary()

history = lstm_model.train(
    lstm, X_trn_seq, y_trn_seq, X_val_seq, y_val_seq,
    epochs=100, batch_size=256,
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history.history['loss'],     label='Train loss')
ax.plot(history.history['val_loss'], label='Val loss')
ax.set_title('Curva de aprendizaje — LSTM')
ax.set_xlabel('Época'); ax.set_ylabel('MSE (escala normalizada)')
ax.legend(); plt.tight_layout()
plt.savefig(FIGURES_DIR / 'lstm_training_curve.png', bbox_inches='tight')
plt.show()

In [ ]:
y_pred_lstm, y_true_lstm, metrics_lstm = lstm_model.evaluate(
    lstm, X_test_seq, y_test_seq, scaler=scaler_lstm, n_cols=n_lstm_cols
)

print('LSTM — métricas globales:')
for k, v in metrics_lstm.items():
    print(f'  {k:12s}: {v}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sidx = np.random.choice(len(y_true_lstm), size=min(500, len(y_true_lstm)), replace=False)
axes[0].scatter(y_true_lstm[sidx], y_pred_lstm[sidx], alpha=0.4, s=20, color='darkorange')
lims = [0, max(y_true_lstm.max(), y_pred_lstm.max()) * 1.05]
axes[0].plot(lims, lims, 'r--', lw=1.5, label='Línea ideal')
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_title(f'Pred vs Real — LSTM  (R²={metrics_lstm["R2"]:.3f})')
axes[0].set_xlabel('Ventas reales ($)'); axes[0].set_ylabel('Ventas predichas ($)')
axes[0].legend()

errors_lstm = y_pred_lstm - y_true_lstm
axes[1].hist(errors_lstm, bins=60, color='darkorange', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', lw=1.5)
axes[1].axvline(metrics_lstm['Bias'], color='navy', linestyle=':', lw=1.5,
                label=f"Bias = {metrics_lstm['Bias']:.3f}")
axes[1].set_title('Distribución del error — LSTM')
axes[1].set_xlabel('Error ($)'); axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'lstm_evaluation.png', bbox_inches='tight')
plt.show()

## 5 · Baseline K-Means — Perfiles de clúster

La media diaria de cada clúster (entrenamiento) se usa como pronóstico estático en el período de prueba.  
Cuantifica cuánto valor añaden los modelos supervisados sobre solo conocer el segmento de la máquina.

In [ ]:
profiles = cluster_model.build_cluster_profiles(train_df, clusters_df)
print('Perfiles de demanda por clúster:')
print(profiles.to_string(index=False))

In [ ]:
y_pred_kmeans, metrics_kmeans = cluster_model.evaluate_cluster_baseline(
    test_df, clusters_df, profiles, strategy='mean'
)
metrics_kmeans_by_cluster = cluster_model.evaluate_by_cluster(
    test_df, clusters_df, profiles, strategy='mean'
)

print('K-Means baseline — métricas globales:')
for k, v in metrics_kmeans.items():
    print(f'  {k:12s}: {v}')
print('\nMétricas por clúster:')
print(metrics_kmeans_by_cluster.to_string(index=False))

In [ ]:
cluster_names = {
    0: 'Clúster 0\nRendimiento moderado',
    1: 'Clúster 1\nAlta venta, bajo surtido',
    2: 'Clúster 2\nAlta frecuencia',
    3: 'Clúster 3\nOutliers (alto valor)',
}

# test_df ya tiene 'cluster' del pipeline — no se necesita merge
cluster_daily = test_df.groupby(['date', 'cluster'])['total_sale'].sum().reset_index()

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig)

for i, (cluster_id, label) in enumerate(cluster_names.items()):
    ax = fig.add_subplot(gs[i // 2, i % 2])
    cdata = cluster_daily[cluster_daily['cluster'] == cluster_id]
    if not cdata.empty:
        ax.plot(cdata['date'], cdata['total_sale'], lw=1.5, label='Real')
        pv = profiles.loc[profiles['cluster'] == cluster_id, 'mean_daily_sale'].values
        if len(pv):
            nm = profiles.loc[profiles['cluster'] == cluster_id, 'n_machines'].values[0]
            ax.axhline(pv[0] * nm, color='red', linestyle='--', lw=1.5, label='Media clúster × n_máq.')
    ax.set_title(label); ax.set_xlabel('Fecha'); ax.set_ylabel('Ventas ($)')
    ax.legend(fontsize=8)

plt.suptitle('Demanda diaria por clúster — período de prueba', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cluster_demand_profiles.png', bbox_inches='tight')
plt.show()

## 6 · Comparación integral de modelos

### 6.1 Tabla de métricas completa

In [ ]:
MODELS = {
    'K-Means (baseline)': metrics_kmeans,
    'XGBoost':            metrics_xgb,
    'LSTM':               metrics_lstm,
}

comparison = pd.DataFrame(MODELS).T.reset_index().rename(columns={'index': 'Modelo'})

# Mejora relativa sobre baseline (para métricas donde menor = mejor)
baseline = MODELS['K-Means (baseline)']
for col in ['MAE', 'RMSE', 'MdAE', 'SMAPE (%)']:
    comparison[f'Δ {col} (%)'] = (
        (baseline[col] - comparison[col]) / baseline[col] * 100
    ).round(2)

# Para R² mayor = mejor → invertir signo de mejora
comparison['Δ R² (%)'] = (
    (comparison['R2'] - baseline['R2']) / abs(baseline['R2']) * 100
).round(2)

display_cols = ['Modelo', 'MAE', 'RMSE', 'MdAE', 'R2', 'SMAPE (%)', 'Bias']
print(comparison[display_cols].to_string(index=False))

comparison.to_csv(METRICS_DIR / 'model_comparison.csv', index=False)
print('\n✓ Guardado en reports/metrics/model_comparison.csv')

### 6.2 Gráficos de barras — seis métricas

In [ ]:
COLORS  = ['#7f8c8d', '#2980b9', '#e67e22']
modelos = comparison['Modelo'].tolist()

# Métricas donde MENOR es mejor
lower_is_better = ['MAE', 'RMSE', 'MdAE', 'SMAPE (%)']
# R² y Bias tienen interpretación distinta

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, (metric, label, fmt) in enumerate([
    ('MAE',       'MAE ($)',         '${:.4f}'),
    ('RMSE',      'RMSE ($)',        '${:.4f}'),
    ('MdAE',      'MdAE ($)',        '${:.4f}'),
    ('R2',        'R²',              '{:.4f}'),
    ('SMAPE (%)', 'SMAPE (%)',       '{:.2f}%'),
    ('Bias',      'Bias ($) \n+ sobreestima / − subestima', '${:.4f}'),
]):
    ax   = axes[i]
    vals = comparison[metric].astype(float).tolist()
    bars = ax.bar(modelos, vals, color=COLORS, edgecolor='white', width=0.5)
    ax.set_title(label, fontsize=11)
    ax.bar_label(bars, labels=[fmt.format(v) for v in vals], padding=3, fontsize=8)
    ax.tick_params(axis='x', rotation=12)
    # Highlight best bar
    if metric in lower_is_better:
        best = int(np.argmin(vals))
    elif metric == 'R2':
        best = int(np.argmax(vals))
    else:            # Bias: closest to 0 is best
        best = int(np.argmin(np.abs(vals)))
    bars[best].set_edgecolor('gold')
    bars[best].set_linewidth(2.5)
    # Bias: add zero line
    if metric == 'Bias':
        ax.axhline(0, color='black', lw=1, linestyle='--')

plt.suptitle('Comparación integral — 6 métricas  (borde dorado = mejor valor)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_comparison_full.png', bbox_inches='tight')
plt.show()

### 6.3 Gráfico de radar — perfil multidimensional por modelo

In [ ]:
# Para el radar normalizamos cada métrica a [0, 1] donde 1 = mejor
radar_metrics = ['MAE', 'RMSE', 'MdAE', 'R2', 'SMAPE (%)']
radar_labels  = ['MAE\n(↓)', 'RMSE\n(↓)', 'MdAE\n(↓)', 'R²\n(↑)', 'SMAPE\n(↓)']

def normalize_for_radar(df, metrics):
    normed = df[metrics].astype(float).copy()
    for m in metrics:
        col = normed[m]
        mn, mx = col.min(), col.max()
        if mx == mn:
            normed[m] = 1.0
        elif m == 'R2':          # higher is better
            normed[m] = (col - mn) / (mx - mn)
        else:                    # lower is better
            normed[m] = 1 - (col - mn) / (mx - mn)
    return normed

normed = normalize_for_radar(comparison, radar_metrics)

N    = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close the polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for idx, row in comparison.iterrows():
    values = normed.iloc[idx].tolist() + [normed.iloc[idx].tolist()[0]]
    ax.plot(angles, values, lw=2, label=row['Modelo'], color=COLORS[idx])
    ax.fill(angles, values, alpha=0.12, color=COLORS[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=7)
ax.set_title('Perfil de desempeño multidimensional\n(mayor área = mejor)', pad=20, fontsize=12)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'radar_comparison.png', bbox_inches='tight')
plt.show()

### 6.4 Heatmap de métricas por clúster — XGBoost vs K-Means

In [ ]:
metrics_xgb_by_cluster['Modelo']    = 'XGBoost'
metrics_kmeans_by_cluster['Modelo'] = 'K-Means'
by_cluster_all = pd.concat([metrics_xgb_by_cluster, metrics_kmeans_by_cluster], ignore_index=True)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, metric in zip(axes, ['MAE', 'RMSE', 'MdAE', 'SMAPE (%)']):
    pivot = by_cluster_all.pivot(index='cluster', columns='Modelo', values=metric)
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn_r',
                ax=ax, cbar_kws={'shrink': 0.7})
    ax.set_title(metric); ax.set_xlabel('Modelo'); ax.set_ylabel('Clúster')

plt.suptitle('Métricas por clúster — XGBoost vs K-Means  (verde = menor error)', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'heatmap_by_cluster.png', bbox_inches='tight')
plt.show()

### 6.5 Distribución de errores comparada

In [ ]:
# Errores XGBoost y K-Means sobre el mismo conjunto (test_clean)
y_true_common  = test_clean[TARGET_COL].values
y_pred_km_clean = cluster_model.predict_cluster_baseline(test_clean, clusters_df, profiles)

err_xgb = y_pred_xgb          - y_true_common
err_km  = y_pred_km_clean      - y_true_common
err_lst = y_pred_lstm          - y_true_lstm    # its own test set

fig, ax = plt.subplots(figsize=(12, 5))
bins = np.linspace(-20, 20, 80)

ax.hist(err_km,  bins=bins, alpha=0.45, label='K-Means',  color=COLORS[0], density=True)
ax.hist(err_xgb, bins=bins, alpha=0.55, label='XGBoost',  color=COLORS[1], density=True)
ax.hist(err_lst, bins=bins, alpha=0.45, label='LSTM',     color=COLORS[2], density=True)

for err, c, m in zip([err_km, err_xgb, err_lst], COLORS,
                      [metrics_kmeans['Bias'], metrics_xgb['Bias'], metrics_lstm['Bias']]):
    ax.axvline(m, color=c, lw=2, linestyle='--')

ax.axvline(0, color='black', lw=1.5)
ax.set_title('Distribución del error de predicción — los tres modelos (Bias = línea discontinua)')
ax.set_xlabel('Error ($)  [predicción − real]')
ax.set_ylabel('Densidad')
ax.legend(); plt.tight_layout()
plt.savefig(FIGURES_DIR / 'error_distribution_comparison.png', bbox_inches='tight')
plt.show()

## 7 · Conclusiones y análisis integral

### 7.1 Contexto del conjunto de datos de evaluación

El período de prueba abarca **95 días** (26-dic-2025 → 30-mar-2026) con **38 475 filas-día** distribuidas sobre 405 máquinas expendedoras en El Paso, TX. Una característica dominante del dataset es su **alta dispersión**: el 61.5 % de las filas registran ventas nulas (máquina sin actividad ese día), con una media de $6.27 y una desviación estándar de $16.43, lo que genera una distribución fuertemente sesgada hacia cero. Este patrón es estructural al negocio y condiciona directamente la capacidad de cualquier modelo para alcanzar R² alto o SMAPE bajo.

---

### 7.2 Perfiles de clúster K-Means

| Clúster | Máquinas | Media diaria | Descripción |
|---|---|---|---|
| 0 — Rendimiento moderado | 182 (44.9 %) | $2.40 | Mayoría de la flota; demanda baja y estable |
| 1 — Alta venta, bajo surtido | 143 (35.3 %) | $3.85 | Segunda categoría más frecuente; pico entre semana |
| 2 — Alta frecuencia | 78 (19.3 %) | $9.88 | Máquinas de alto rendimiento; 4× la media del C0 |
| 3 — Outliers (alto valor) | 2 (0.5 %) | $0.97 | Solo 2 unidades; comportamiento atípico, no representativo |

Los clústeres 0 y 1 concentran el 80 % de la flota y determinan el comportamiento agregado. El clúster 2 aporta desproporcionadamente a los ingresos totales a pesar de representar solo el 19 % de las máquinas. El clúster 3 (2 máquinas) actúa como outlier estadístico y reduce la confiabilidad de cualquier predicción basada exclusivamente en su perfil.

---

### 7.3 Resultados globales comparados

| Modelo | MAE ↓ | RMSE ↓ | MdAE ↓ | R² ↑ | SMAPE ↓ | Bias |
|---|---|---|---|---|---|---|
| K-Means (baseline) | 7.39 | 16.07 | 3.85 | 0.043 | 157.0 % | −1.93 |
| XGBoost | 5.78 | 13.29 | 2.30 | 0.346 | 148.0 % | −0.60 |
| **LSTM** | **5.32** | **12.74** | **1.66** | **0.399** | **149.3 %** | **−0.43** |

**LSTM obtiene el mejor resultado en 5 de 6 métricas.** Respecto al baseline K-Means:
- Reducción de MAE: **28 %** (LSTM) vs 21.7 % (XGBoost)
- Reducción de MdAE: **56.9 %** (LSTM) vs 40.3 % (XGBoost)
- R² pasa de 0.043 a **0.399** (+823 % relativo), lo que significa que el LSTM explica casi 10 veces más varianza que el baseline estático

> **Nota sobre SMAPE ≈ 148–157 %**: Este valor elevado es consecuencia directa del 61.5 % de días con ventas nulas. Cuando el valor real es cero, cualquier predicción positiva produce SMAPE = 200 %. El SMAPE no es la métrica principal de decisión en este contexto; MAE, MdAE y Bias son más interpretables operacionalmente.

---

### 7.4 Análisis por clúster (XGBoost vs K-Means)

| Clúster | MAE XGB | MAE K-Means | R² XGB | Interpretación |
|---|---|---|---|---|
| 0 — Moderado | 4.27 | 4.79 | 0.19 | XGBoost mejora 10.9 %; R² bajo por alta proporción de días cero |
| 1 — Alta venta | 3.82 | 5.69 | 0.46 | XGBoost mejora 32.8 %; **mejor ajuste relativo** de la flota |
| 2 — Alta frecuencia | 13.03 | 16.74 | 0.29 | XGBoost mejora 22.2 %; error absoluto mayor por volumen de ventas más alto |
| 3 — Outliers | 1.30 | 0.97 | −∞ | K-Means gana por azar; solo 2 máquinas, 190 filas — no estadísticamente significativo |

El clúster 1 es donde los modelos supervisados aportan mayor valor: la variabilidad intrasemana (capturada por , , ) no es recuperable desde un perfil estático. El clúster 2, aunque muestra el mayor error absoluto, refleja simplemente que sus máquinas venden más; en términos relativos la mejora es comparable. El clúster 3 no debe usarse como criterio de selección de modelo por su tamaño mínimo.

---

### 7.5 Análisis del sesgo (Bias)

Todos los modelos **subestiman** sistemáticamente (Bias < 0), lo que en el contexto de vending implica riesgo de **quiebre de inventario** (understock). El LSTM minimiza este riesgo con Bias = −0.43, frente a −1.93 del baseline. La causa raíz es la asimetría del dataset: los picos de venta (cola derecha con valores hasta $521.75) son infrecuentes y difíciles de anticipar, por lo que los modelos aprenden a ser conservadores.

**Implicación operativa:** Para rutas de reabastecimiento críticas (clúster 2), se recomienda aplicar un ajuste de +$0.43 sobre las predicciones LSTM o evaluar cuantiles superiores (p75) en lugar de la media como predicción de stock mínimo.

---

### 7.6 Fortalezas y limitaciones por modelo

| Dimensión | K-Means | XGBoost | LSTM |
|---|---|---|---|
| Tipo | No supervisado | Supervisado tabular | Supervisado secuencial |
| Características | Perfil estático del clúster | 12: lags, rolling, temporal, clúster | Secuencias de 14 días × 4 variables |
| Interpretabilidad | Alta | Alta (feature importance) | Baja (caja negra) |
| Velocidad de inferencia | Instantánea | Rápida (~ms) | Moderada (~100 ms/lote) |
| Requiere historial continuo | No | Sí (lag_14 mínimo) | Sí (14 días mínimo) |
| Nuevo vm_control | Asignar clúster | Requiere reentrenamiento | Requiere reentrenamiento |
| Actualización incremental | No aplica | Fácil (warm-start) | Costosa (reentrenamiento completo) |

---

### 7.7 Recomendaciones

1. **Modelo de producción principal → LSTM**: menor MAE, MdAE y bias; captura patrones temporales que los árboles no pueden modelar secuencialmente.

2. **Modelo de respaldo / interpretabilidad → XGBoost**: para auditorías, explicaciones a operaciones, o cuando el historial de una máquina es insuficiente para 14 días de contexto.

3. **K-Means**: mantener como herramienta operacional para **segmentación de rutas** y para asignar predicciones a máquinas nuevas sin historial, no como modelo predictivo primario.

4. **Estrategia anti-quiebre**: sumar el valor absoluto del Bias del LSTM (+0.43) al pronóstico diario para rutas de clúster 2 (alta frecuencia), donde el costo del quiebre es mayor.

5. **Monitoreo continuo**: recalibrar LSTM trimestralmente o ante eventos de deriva (cambio de ubicación de máquinas, nuevos contratos). El XGBoost puede actualizarse con warm-start mensualmente a bajo costo.

6. **Mejoras futuras de mayor impacto**: (a) incluir variables externas —temperatura, días festivos locales de El Paso, eventos— para reducir el residuo del 60 % de varianza no explicada; (b) modelar explícitamente la probabilidad de venta cero (modelo de dos etapas: clasificador + regresor) para reducir el Bias en días sin actividad.


## 8 · Persistencia de artefactos

In [ ]:
import joblib

xgb_model.save_model(xgb, str(ARTIFACTS_DIR / 'xgboost' / 'xgb_demand.json'))
lstm_model.save_model(lstm, str(ARTIFACTS_DIR / 'lstm' / 'lstm_demand.keras'))
joblib.dump(scaler_lstm, str(ARTIFACTS_DIR / 'lstm' / 'lstm_scaler.pkl'))
profiles.to_csv(ARTIFACTS_DIR / 'clustering' / 'cluster_profiles.csv', index=False)

pred_export = test_clean[['vm_control', 'date', TARGET_COL, 'cluster']].copy().reset_index(drop=True)
pred_export['pred_xgb']    = y_pred_xgb
pred_export['pred_kmeans'] = cluster_model.predict_cluster_baseline(test_clean, clusters_df, profiles)
pred_export.to_csv(EXPORTS_DIR / 'test_predictions.csv', index=False)

print('Artefactos guardados ✓')
for p in [
    ARTIFACTS_DIR / 'xgboost' / 'xgb_demand.json',
    ARTIFACTS_DIR / 'lstm'    / 'lstm_demand.keras',
    ARTIFACTS_DIR / 'lstm'    / 'lstm_scaler.pkl',
    ARTIFACTS_DIR / 'clustering' / 'cluster_profiles.csv',
    EXPORTS_DIR   / 'test_predictions.csv',
    METRICS_DIR   / 'model_comparison.csv',
]:
    print(f'  {p}')